# Load an MLP model and convert it to ONNX file
Please take the following steps before running this notebook
1. clone the repo by running `git clone https://github.com/abidihaider/RealTimeAlignment.git`
2. check to the develop branch of the repo
3. run `python setup.py develop`

In [ ]:
import os
import sys
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rich import print as rprint

import torch
from torch import nn
import onnx
import onnxruntime

from rtal.datasets.dataset import ROMDataset
from torch.utils.data import DataLoader

from mlp import MLP

In [ ]:
device = 'cpu'
onnx_folder = Path('onnx_files_narrow_but_deep_untrained/')

## Load some data

In [ ]:
data_root = 'data/rom_det-3_part-200_cont-and-rounded_excerpt/'
dataset  = ROMDataset(data_root, split='train', num_particles=50)

## Load model configuration and use it to initialize models
For the half model, we will cast it to half precision for inference.

In [ ]:
with open('checkpoints/config_narrow_but_deep.yaml', 'r', encoding='UTF-8') as handle:
    config = yaml.safe_load(handle)

model_full = MLP(**config['model'])
model_half = MLP(**config['model']).half()

In [ ]:
# ckpt_path = 'checkpoints/ckpt_last_small.pth'
# ckpt = torch.load(ckpt_path, map_location='cpu')
# model_state_dict = ckpt['model']

# model_full.load_state_dict(model_state_dict)
# model_half.load_state_dict(model_state_dict)
# # the following operation will cast a model's 
# # parameters recursively to half precision in place!
# model_half.half()

## Half-precision inference
Does it significantly affect performance?

Based on the plot, I would say half-precision inference works!

In [ ]:
# Load an event and get the readout
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)
event = next(iter(dataloader))
readout = event[f'readout_curr_cont'].to(device)
readout = torch.transpose(readout, 1, 2).flatten(-2, -1)

In [ ]:
# infer with full precision
output_full = model_full(readout)

# infer with half precision
output_half = model_half(readout.half())

# difference between the two outputs
diff = (output_half.to(torch.float32) - output_full).flatten().detach().cpu().numpy()
mean = diff.mean()

In [ ]:
fig, ax = plt.subplots(1, 1)
_ = ax.hist(diff, bins=50)
ax.axvline(mean, label=f'mean = {mean:.8f}', color='orange')
ax.legend()

## Convert the full models to an ONNX file
Notice the model size is halved for the torch.float16 model

In [ ]:
# Create dummy input with the correct shape
in_features = config['model']['in_features']
num_particles = config['data']['num_particles']

dummy_input = torch.randn(1, num_particles, in_features)

model_dict = {'full': model_full,
              'half': model_half}

# Export to ONNX
for precision, model in model_dict.items():
    
    if precision == 'half':
        model_input = dummy_input.half()
    else:
        model_input = dummy_input

    onnx_path = onnx_folder/f'mlp_{precision}.onnx'
    
    torch.onnx.export(
        model,                               # model being run
     
        model_input,                         # model input (or a tuple for multiple inputs)
        onnx_path,                           # where to save the model (filename)
        export_params       = True,          # store the trained weights inside the model
        opset_version       = 11,            # the ONNX version to export to (11 is widely supported)
        do_constant_folding = True,          # optimize constants
        input_names         = ['input'],     # input name (can be arbitrary)
        output_names        = ['output'],    # output name
        # support dynamic batch size
        dynamic_axes        = {'input'  : {0: 'batch_size'},
                               'output' : {0: 'batch_size'}}
    )
    print(f'\n{precision} model has been exported to ONNX format.')
    
    onnx_size = os.path.getsize(onnx_path)
    print(f'Size of {onnx_path} is {onnx_size} bytes.')

## Load the ONNX model and check its validity

In [ ]:
for precision, model in model_dict.items():
    onnx_path = onnx_folder/f'mlp_{precision}.onnx' 
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)

    print(f'{precision} ONNX model is valid!')

## Test the ONNX model with ONNX Runtime

In [ ]:
for precision, model in model_dict.items():

    onnx_path = onnx_folder/f'mlp_{precision}.onnx' 
    
    ort_session = onnxruntime.InferenceSession(onnx_path)
    
    # Convert to dummy input to numpy
    if precision == 'half':
        input_data = dummy_input.numpy().astype(np.float16)
    else:
        input_data = dummy_input.numpy()
        
    # Run the ONNX model
    ort_inputs = {ort_session.get_inputs()[0].name: input_data}
    ort_outputs = ort_session.run(None, ort_inputs)
    
    print(f"{precision} ONNX model tested successfully!")

## Test the ONNX model and their PyTorch correspondence produce the same resutls
**The half ONNX model and half PyTorch model do not produce exactly the same results!!!!**

In [ ]:
# Get one batch containing 10 events (since we set the `batch_size=10` 
# in the data loader) and run PyTorch/ONNX models on them to see
# whether they produce the same results. 

dataloader = DataLoader(dataset, batch_size=10, shuffle=False)
event = next(iter(dataloader))
# readout generated by the misaligned detectors
# "_cont" here means the coordinates are continuous without any rounding.
readout = event[f'readout_curr_cont'].to(device)
# readout: (batch_size, num_detectors, num_particles, 2)
#        ->(batch_size, num_particles, num_detectors, 2)
#        ->(batch_size, num_particles, num_detectors x 2)
readout = torch.transpose(readout, 1, 2).flatten(-2, -1)

In [ ]:
def run_onnx(session, inputs):
    if isinstance(session, str):
        ort_session = onnxruntime.InferenceSession(session)
    elif isinstance(session, onnxruntime.capi.onnxruntime_inference_collection.InferenceSession):
        ort_session = session
    else:
        raise ValueError('session should either be a str (path to a onnx file) or a onnxruntime session')
    
    ort_inputs = {ort_session.get_inputs()[0].name: inputs}
    
    return ort_session.run(None, ort_inputs)[0]   

In [ ]:
for precision, model in model_dict.items():

    # get torch outputs
    if precision == 'half':
        torch_inputs = readout.half()
    else:
        torch_inputs = readout
    
    torch_outputs = model(torch_inputs)

    # get ONNX outputs
    if precision == 'half':
        onnx_inputs = readout.numpy().astype(np.float16)
    else:
        onnx_inputs = readout.numpy()

    onnx_path = onnx_folder/f'mlp_{precision}.onnx' 
    ort_outputs = run_onnx(str(onnx_path), onnx_inputs)

    # Compare PyTorch and ONNX outputs
    difference = (torch_outputs.detach().numpy() - ort_outputs).flatten()
    max_diff = np.max(np.abs(difference))
    
    print(f"Max difference between {precision} PyTorch and ONNX outputs: {max_diff:.6f}")

    if precision == 'half':
        l1 = np.abs(difference).mean()
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        _ = ax.hist(difference, bins=10)
        ax.axvline(mean, label=f'L1 = {l1:.8f}', color='orange')
        ax.legend()
        ax.set_xlabel('difference')
        ax.set_ylabel('count')
        ax.set_title(f'Distribution of difference between half ONNX and PyTorch models')

## Convert submodules to ONNX files for the half model

### let us first see what submodels we have

In [ ]:
for name, module in model_half.named_children():
    rprint(f'[bold green]== {name} ==================================[/bold green]')
    print(module)

### Create dummy input for all submodels with the correct shape

In [ ]:
num_particles = config['data']['num_particles']

in_features = config['model']['in_features']
embedding_features = config['model']['embedding_features']
# Note that the subset solvers all have the same config,
# and hence getting the configuration of the first one is enough.
num_solvers = len(config['model']['subset_config'])
subset_size = config['model']['subset_config'][0][0]
subset_features = config['model']['subset_config'][0][-1]

dummy_embed = torch.randn(1, num_particles, in_features)
rprint(f'[green]embed[/green] submodule input shape {dummy_embed.shape}')
dummy_solvers = torch.randn(1, num_particles, subset_size * embedding_features[-1])
rprint(f'[green]solver[/green] submodule input shape {dummy_solvers.shape}')
dummy_output = torch.randn(1, num_particles, subset_features)
rprint(f'[green]output[/green] submodule input shape {dummy_output.shape}')

### Get all submodules

In [ ]:
submodule_embed = model_half.get_submodule('embed')
submodule_solvers = [mod for mod in model_half.get_submodule('solvers')]
submodule_output = model_half.get_submodule('output')

### Let the conversion begin! 

In [ ]:
torch.onnx.export(
    submodule_embed,
    dummy_embed.half(),
    onnx_folder/"submodule_embed.onnx",
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'},
    }
)

for i, mod in enumerate(submodule_solvers):
    torch.onnx.export(
        mod.model,
        dummy_solvers.half(),
        onnx_folder/f"submodule_solvers-{i}.onnx",
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'},
        }
    )

torch.onnx.export(
    submodule_output,
    dummy_output.half(),
    onnx_folder/"submodule_output.onnx",
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'},
    }
)

## <span style="color:blue">Test whether submodules produce the same results</span>
against the ONNX module converted as single piece.

In [ ]:
# Load ONNX submodules
session_embed = onnxruntime.InferenceSession(onnx_folder/'submodule_embed.onnx')
session_solvers = []
for solver_id in range(num_solvers):
    session = onnxruntime.InferenceSession(onnx_folder/f'submodule_solvers-{solver_id}.onnx')
    session_solvers.append(session)
session_output = onnxruntime.InferenceSession(onnx_folder/'submodule_output.onnx')

### run submodules onnx

In [ ]:
def assemble_np(array, subset_size):
    """
    Function to assemble the subset
    """
    return np.concatenate([np.roll(array, shift=i, axis=1) 
                           for i in range(subset_size)], 
                          axis=-1)
# data
onnx_inputs = readout.numpy().astype(np.float16)
# np.savetxt('onnx_txt/embed_input.txt', onnx_inputs[event_id].flatten(), fmt="%.8g")

# run embedding
embed_outputs = run_onnx(session_embed, onnx_inputs)
# np.savetxt('onnx_txt/embed_output.txt', embed_outputs[event_id].flatten(), fmt="%.8g")

# run subset solvers
array = embed_outputs
for session in session_solvers:
    array = assemble_np(array, subset_size)
    array = run_onnx(session, array)

# run output
outputs_sub = run_onnx(session_output, array)
outputs_sub = outputs_sub.mean(axis=1)

### run whole module onnx

In [ ]:
outputs_whole = run_onnx(str(onnx_folder/'mlp_half.onnx'), onnx_inputs)

### Distribution of the difference
**NOTE:** it seems that running submodules one-by-one will produce slightly different results than running the model as a whole.
But I think the differenc is small enough to be ignored. 

In [ ]:
difference = (outputs_sub - outputs_whole).flatten()
l1 = np.abs(difference).mean()
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
_ = ax.hist(difference, bins=10)
ax.axvline(mean, label=f'L1 = {l1:.8f}', color='orange')
ax.legend()
ax.set_xlabel('difference')
ax.set_ylabel('count')
ax.set_title(f'Distribution of difference between half ONNX and PyTorch models')